# 23 - Form scope across competitions, and rotation

Which matches should a club's form be built from (docs/multi-league-plan.md
§4B.5, decision D9)? Today each feature only sees the competition being
predicted. The grid, separately for **domestic** and **CL** predictions:

| Scope | `XGDDiffLast5` from | `GoalsPerGameDiff` from |
|---|---|---|
| **S1** domestic-only (today's domestic) | last 5 league games | league, season to date |
| **S2** all competitions | last 5 games in any competition (league + CL/EL/ECL) | all competitions, season to date |
| **S3** same competition (today's CL) | last 5 games in the competition predicted | that competition, season to date |
| **S4** all, opponent-adjusted | as S2, each game's xGD + (R_opp - 1500)/c | - |

4 xGD scopes x 3 goals scopes = 12 combinations per target. For domestic
matches S3 is S1, so 6 are distinct there. `c` is tuned on 22-23/23-24.

**Rules (§5, D9).**
- **Domestic:** the shipped models (E0 on its own, the rest pooled). A combination
  other than S1/S1 ships only if it is **better** than S1/S1 (CI entirely below 0)
  on all domestic test matches. The CL-club subset is reported too.
- **CL:** notebook 22's chosen setup (c), the Euro-only LR. The best combination
  ships if its CI against S2/S1 (the prior favourite) is clear of zero; otherwise
  the simplest does. Setup (a) is also shown, in case the owner ships (a).
- **Rotation** (`RestDaysDiff`, `PostEuroMidweek`) are domestic challengers,
  judged with the domestic rule.

Walk-forward by season, test 22-23 to 25-26, paired bootstrap (5,000, seed 42).
Elo is the shipped shared-scale `EloDiff` throughout; only the form features change.

In [1]:
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from elo import shared_scale
from european import EUROPEAN, results_for_elo
from evaluate import better, paired_bootstrap, per_match_log_loss, seasons_better
from features import build_history
from load_data import build_cup_rows, read_matches
from model import FEATURES, LEAGUE_COLUMNS, OWN_MODEL, league_columns, make_model, walk_forward_models

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

TEST = ["22-23", "23-24", "24-25", "25-26"]
TUNE, REPORT = TEST[:2], TEST[2:]
LAST = TEST[-1]
LABELS = ["A", "D", "H"]

## Data

As notebook 22: the 7 leagues and the CL from `all_matches.csv`, the EL and
ECL from the FotMob caches, features from the pipeline's `build_history`
(shared-scale Elo; form scoped to the competition, i.e. S1 for league rows
and S3 for European rows).

In [2]:
matches = read_matches(cups=True)
domestic = matches[~matches["Div"].isin(EUROPEAN)]
el_ecl = build_cup_rows(domestic, ["EL", "ECL"])
history = pd.concat([matches, el_ecl.reindex(columns=matches.columns)], ignore_index=True)
history = history[history["Season"] <= LAST]

european = results_for_elo(domestic)
european = european[european["Season"] <= LAST].reset_index(drop=True)
data, _, _ = build_history(history, european=european)
data["Euro"] = data["Div"].isin(EUROPEAN).astype(int)
data["MatchKey"] = np.arange(len(data))
print(data.groupby("Euro").size().to_dict(), "rows (league, European)")

{0: 11968, 1: 2280} rows (league, European)


**Opponent strength for S4**: every club's shared-scale rating going into each
of its matches. `shared_scale(history_from=...)` records each club's rating
after every match; a club's rating going into a match is its rating after its
previous one (a club new to the record starts at its league's level, taken as
1500 here - it only matters for a club's first game).

In [3]:
dom_sorted = data[data["Euro"] == 0].sort_values(["MatchDateTime", "HomeTeam", "AwayTeam"])
after = shared_scale(dom_sorted, european, history_from=data["MatchDateTime"].min())["history"]
after = after.sort_values("Date", kind="stable")
# Several records on one day only for the same match: keep the last
after = after.drop_duplicates(["Club", "Date"], keep="last")
print(f"{len(after)} club-match ratings recorded")

28492 club-match ratings recorded


## One row per club per match

The form features, every scope, from one table of appearances: club, kick-off,
competition, xG for/against, goals for/against, the opponent's rating going in.

In [4]:
def appearances(frame):
    sides = []
    for side, other, gf, ga, xf, xa in [("HomeTeam", "AwayTeam", "FTHG", "FTAG", "HomeXG", "AwayXG"),
                                        ("AwayTeam", "HomeTeam", "FTAG", "FTHG", "AwayXG", "HomeXG")]:
        sides.append(pd.DataFrame({
            "MatchKey": frame["MatchKey"], "When": frame["MatchDateTime"], "Day": frame["MatchDateTime"].dt.normalize(),
            "Season": frame["Season"], "Div": frame["Div"], "Euro": frame["Euro"],
            "Club": frame[side], "Opponent": frame[other], "Side": side[:4],
            "GF": frame[gf], "GA": frame[ga], "XGD": frame[xf] - frame[xa],
        }))
    return pd.concat(sides, ignore_index=True)


apps = appearances(data)

# Opponent's rating going in = its rating after its last match before this day
ratings = after.rename(columns={"Club": "Opponent", "Date": "RatedOn"})[["Opponent", "RatedOn", "Rating"]]
ratings["RatedOn"] = pd.to_datetime(ratings["RatedOn"])
apps = pd.merge_asof(
    apps.sort_values("Day"), ratings.sort_values("RatedOn"),
    left_on="Day", right_on="RatedOn", by="Opponent", allow_exact_matches=False, direction="backward",
).rename(columns={"Rating": "OppRating"})
apps["OppRating"] = apps["OppRating"].fillna(1500)
print(len(apps), "appearances")

28496 appearances


### Features as of kick-off

For a scope (a subset of appearances), each club's last-5 xGD and season goals
per game **after** each game; a match then takes, for each side, the latest
value from strictly before its kick-off. The last-5 window counts games
without xG but averages only the ones with it (0 if none), and goals per game
is 0 before a club's first game of the season - both as `features.py` does.

In [5]:
def after_each_game(subset, xgd_column="XGD"):
    subset = subset.sort_values(["Club", "When"]).copy()
    by_club = subset.groupby("Club", sort=False)
    subset["Last5"] = by_club[xgd_column].transform(lambda s: s.rolling(5, min_periods=1).mean()).fillna(0)
    by_season = subset.groupby(["Club", "Season"], sort=False)
    subset["GPG"] = by_season["GF"].cumsum() / (by_season.cumcount() + 1)
    return subset[["Club", "When", "Season", "Last5", "GPG"]]


def as_of(targets, state):
    """Each target appearance's Last5 / season GPG from strictly before its kick-off."""

    looked_up = pd.merge_asof(
        targets.sort_values("When")[["MatchKey", "Side", "Club", "When", "Season"]],
        state.rename(columns={"When": "StateWhen", "Season": "StateSeason"}).sort_values("StateWhen"),
        left_on="When", right_on="StateWhen", by="Club", allow_exact_matches=False, direction="backward",
    )
    looked_up["Last5"] = looked_up["Last5"].fillna(0)
    looked_up["GPG"] = looked_up["GPG"].where(looked_up["StateSeason"] == looked_up["Season"], 0).fillna(0)
    return looked_up


def diff(looked_up, column):
    wide = looked_up.pivot_table(index="MatchKey", columns="Side", values=column)
    return (wide["Home"] - wide["Away"]).reindex(data["MatchKey"]).to_numpy()


def scope_features(c=None):
    """{scope: (XGDDiffLast5, GoalsPerGameDiff)} for every row of ``data``."""

    out = {}
    pools = {"S1": apps[apps["Euro"] == 0], "S2": apps}
    for scope, pool in pools.items():
        looked = as_of(apps, after_each_game(pool))
        out[scope] = (diff(looked, "Last5"), diff(looked, "GPG"))

    # S3: each match's own competition
    looked = pd.concat([as_of(apps[apps["Div"] == d], after_each_game(apps[apps["Div"] == d]))
                        for d in apps["Div"].unique()])
    out["S3"] = (diff(looked, "Last5"), diff(looked, "GPG"))
    return out


def s4_xgd(c):
    adjusted = apps.assign(XGDAdj=apps["XGD"] + (apps["OppRating"] - 1500) / c)
    looked = as_of(apps, after_each_game(adjusted, "XGDAdj"))
    return diff(looked, "Last5")


SCOPES = scope_features()

**Check against the pipeline.** S1 on league rows and S3 on European rows are
what `build_history` computes; they must match it.

In [6]:
is_euro = data["Euro"].to_numpy() == 1
for scope, rows, label in [("S1", ~is_euro, "league rows, S1"), ("S3", is_euro, "European rows, S3")]:
    xgd, gpg = SCOPES[scope]
    print(f"{label}: max |XGDDiffLast5 - pipeline| = {np.abs(xgd[rows] - data['XGDDiffLast5'].to_numpy()[rows]).max():.2e}, "
          f"max |GoalsPerGameDiff - pipeline| = {np.abs(gpg[rows] - data['GoalsPerGameDiff'].to_numpy()[rows]).max():.2e}")

league rows, S1: max |XGDDiffLast5 - pipeline| = 1.67e-15, max |GoalsPerGameDiff - pipeline| = 0.00e+00
European rows, S3: max |XGDDiffLast5 - pipeline| = 1.11e-15, max |GoalsPerGameDiff - pipeline| = 0.00e+00


## Domestic grid

The shipped setup (`walk_forward_models`: E0 on its own model, the other six
pooled with league indicators), with league rows' form swapped for each
combination. All domestic test matches, and the subset involving a club that
played in Europe that season.

In [7]:
dom = league_columns(data[data["Euro"] == 0].copy())
dom_index = dom.index.to_numpy()
dom_test = dom[dom["Season"].isin(TEST)]

euro_clubs = set(zip(data.loc[is_euro, "Season"], data.loc[is_euro, "HomeTeam"])) | \
             set(zip(data.loc[is_euro, "Season"], data.loc[is_euro, "AwayTeam"]))
cl_club = np.array([(s, h) in euro_clubs or (s, a) in euro_clubs
                    for s, h, a in zip(dom_test["Season"], dom_test["HomeTeam"], dom_test["AwayTeam"])])


def domestic_proba(xgd, gpg, extra=None):
    frame = dom.copy()
    frame["XGDDiffLast5"] = xgd[dom_index]
    frame["GoalsPerGameDiff"] = gpg[dom_index]
    if extra:
        import model as model_module
        saved = list(model_module.FEATURES)
        model_module.FEATURES[:] = saved + extra
        try:
            return model_module.walk_forward_models(frame, TEST)
        finally:
            model_module.FEATURES[:] = saved
    return walk_forward_models(frame, TEST)


def score(rows, candidate, reference, mask=None):
    y = rows["FTR"]
    cand = per_match_log_loss(y, candidate.loc[rows.index][LABELS], LABELS)
    ref = per_match_log_loss(y, reference.loc[rows.index][LABELS], LABELS)
    if mask is not None:
        y, cand, ref = y[mask], cand[mask], ref[mask]
    boot = paired_bootstrap(cand - ref)
    return {"log loss": cand.mean(), "diff": boot["mean"], "CI low": boot["ci"][0], "CI high": boot["ci"][1],
            "better": better(boot["ci"])}


# S4's c: tuned on the tune seasons with GoalsPerGame S1
C_GRID = [100, 200, 400, 800]
S4 = {c: s4_xgd(c) for c in C_GRID}

dom_ref = domestic_proba(*SCOPES["S1"])
tune_rows = dom_test[dom_test["Season"].isin(TUNE)]
c_tune = {c: score(tune_rows, domestic_proba(S4[c], SCOPES["S1"][1]), dom_ref)["diff"] for c in C_GRID}
C_DOMESTIC = min(c_tune, key=c_tune.get)
print("S4 c, domestic tune seasons (diff vs S1/S1):", {c: round(v, 5) for c, v in c_tune.items()}, "-> c =", C_DOMESTIC)

S4 c, domestic tune seasons (diff vs S1/S1): {100: 0.00222, 200: 0.00015, 400: -0.00046, 800: -0.00053} -> c = 800


In [8]:
def xgd_scope(name, c):
    return S4[c] if name == "S4" else SCOPES[name][0]


dom_rows = []
dom_proba = {}
for xs in ["S1", "S2", "S3", "S4"]:
    for gs in ["S1", "S2", "S3"]:
        label = f"{xs}/{gs}"
        distinct = f"{'S1' if xs == 'S3' else xs}/{'S1' if gs == 'S3' else gs}"
        if distinct not in dom_proba:
            dom_proba[distinct] = domestic_proba(xgd_scope(xs, C_DOMESTIC), SCOPES[gs][1])
        proba = dom_proba[distinct]
        everything = score(dom_test, proba, dom_ref)
        subset = score(dom_test, proba, dom_ref, mask=cl_club)
        report = score(dom_test[dom_test["Season"].isin(REPORT)], proba, dom_ref)
        dom_rows.append({
            "xGD/GPG": label, "same as": distinct if distinct != label else "",
            "log loss": everything["log loss"], "diff vs S1/S1": everything["diff"],
            "CI low": everything["CI low"], "CI high": everything["CI high"], "better": everything["better"],
            "seasons better": seasons_better(dom_test["FTR"], proba.loc[dom_test.index][LABELS],
                                             dom_ref.loc[dom_test.index][LABELS], dom_test["Season"], LABELS),
            "Euro-club subset diff": subset["diff"], "subset CI high": subset["CI high"],
            "report-seasons diff": report["diff"],
        })

domestic_grid = pd.DataFrame(dom_rows).set_index("xGD/GPG")
print(f"domestic test matches {len(dom_test)}, involving a European club {cl_club.sum()}")
print(domestic_grid.round(5).to_string())

domestic test matches 9530, involving a European club 5466
        same as  log loss  diff vs S1/S1   CI low  CI high  better  seasons better  Euro-club subset diff  subset CI high  report-seasons diff
xGD/GPG                                                                                                                                       
S1/S1             0.97795        0.00000  0.00000  0.00000   False               0                0.00000         0.00000              0.00000
S1/S2             0.97799        0.00005 -0.00021  0.00030   False               2               -0.00004         0.00037              0.00009
S1/S3     S1/S1   0.97795        0.00000  0.00000  0.00000   False               0                0.00000         0.00000              0.00000
S2/S1             0.97812        0.00017 -0.00057  0.00092   False               1                0.00016         0.00142              0.00084
S2/S2             0.97821        0.00026 -0.00057  0.00107   False               1 

## Rotation (domestic challengers)

- `RestDaysDiff`: days since each side's last match in any competition we hold
  (league, CL, EL, ECL; domestic cups aren't in the data), capped at 7, home
  minus away.
- `PostEuroMidweek`: 1 if a side played a European match in the previous 4 days,
  home minus away.

Added to the shipped features (S1/S1), against S1/S1.

In [9]:
ordered = apps.sort_values(["Club", "When"])
ordered["PrevDay"] = ordered.groupby("Club")["Day"].shift(1)
ordered["PrevEuroDay"] = ordered["Day"].where(ordered["Euro"] == 1).groupby(ordered["Club"]).transform(
    lambda s: s.shift(1).ffill())
ordered["Rest"] = ((ordered["Day"] - ordered["PrevDay"]).dt.days).clip(upper=7).fillna(7)
ordered["PostEuro"] = ((ordered["Day"] - ordered["PrevEuroDay"]).dt.days <= 4).astype(int)

wide = ordered.pivot_table(index="MatchKey", columns="Side", values=["Rest", "PostEuro"])
data["RestDaysDiff"] = (wide[("Rest", "Home")] - wide[("Rest", "Away")]).reindex(data["MatchKey"]).to_numpy()
data["PostEuroMidweek"] = (wide[("PostEuro", "Home")] - wide[("PostEuro", "Away")]).reindex(data["MatchKey"]).to_numpy()
dom[["RestDaysDiff", "PostEuroMidweek"]] = data.loc[dom.index, ["RestDaysDiff", "PostEuroMidweek"]]

print(dom_test.assign(PostEuro=dom.loc[dom_test.index, "PostEuroMidweek"] != 0)
      .groupby("PostEuro").size().rename("domestic test matches"))

rotation_rows = []
for extra in (["RestDaysDiff"], ["PostEuroMidweek"], ["RestDaysDiff", "PostEuroMidweek"]):
    proba = domestic_proba(*SCOPES["S1"], extra=extra)
    everything = score(dom_test, proba, dom_ref)
    subset = score(dom_test, proba, dom_ref, mask=cl_club)
    rotation_rows.append({"added": " + ".join(extra), **everything,
                          "seasons better": seasons_better(dom_test["FTR"], proba.loc[dom_test.index][LABELS],
                                                           dom_ref.loc[dom_test.index][LABELS], dom_test["Season"], LABELS),
                          "Euro-club subset diff": subset["diff"], "subset CI": (round(subset["CI low"], 5), round(subset["CI high"], 5))})
rotation = pd.DataFrame(rotation_rows).set_index("added")
print(rotation.round(5).to_string())

PostEuro
False    8186
True     1344
Name: domestic test matches, dtype: int64


                                log loss     diff   CI low  CI high  better  seasons better  Euro-club subset diff            subset CI
added                                                                                                                                  
RestDaysDiff                     0.97808  0.00014 -0.00042  0.00072   False               2                0.00063  (-0.00022, 0.00149)
PostEuroMidweek                  0.97814  0.00019 -0.00022  0.00062   False               2                0.00047  (-0.00023, 0.00117)
RestDaysDiff + PostEuroMidweek   0.97807  0.00013 -0.00050  0.00076   False               1                0.00060  (-0.00032, 0.00151)


## CL grid

Notebook 22's setup (c): an LR on European rows only (CL + EL + ECL), features
`EloDiff`, `XGDDiffLast5`, `GoalsPerGameDiff`, `Comp_EL`, `Comp_ECL`,
`Knockout`, neutral venues both ways round. European rows' form is swapped for
each combination (the league rows (a) trains on keep S1/S1). Scored on the CL
test matches (628) and all European test matches, against S2/S1.

In [10]:
for div in EUROPEAN:
    data[f"Comp_{div}"] = (data["Div"] == div).astype(int)
data["Knockout"] = data["Stage"].isin({"KO_PLAYOFF", "R16", "QF", "SF", "FINAL"}).astype(int)
data = league_columns(data)
INDICATORS = [c for c in LEAGUE_COLUMNS.values() if c in data.columns]

C_FEATURES = FEATURES + ["Comp_EL", "Comp_ECL", "Knockout"]
A_FEATURES = FEATURES + INDICATORS
ANTISYMMETRIC = ["EloDiff", "XGDDiffLast5", "GoalsPerGameDiff"]
EURO = data["Euro"] == 1
DOMESTIC = ~EURO
euro_test = data[EURO & data["Season"].isin(TEST)]
cl_test = euro_test[euro_test["Div"] == "CL"]


def predict(model, frame, features):
    proba = pd.DataFrame(model.predict_proba(frame[features]), index=frame.index, columns=model.classes_)[LABELS]
    neutral = frame["Neutral"].fillna(0).astype(int) == 1
    if neutral.any():
        flipped = frame.loc[neutral, features].copy()
        flipped[ANTISYMMETRIC] *= -1
        reverse = pd.DataFrame(model.predict_proba(flipped), index=flipped.index, columns=model.classes_)
        for a, b in [("H", "A"), ("D", "D"), ("A", "H")]:
            proba.loc[neutral, a] = (proba.loc[neutral, a] + reverse[b]) / 2
    return proba


def european_proba(xgd, gpg, setup="c"):
    frame = data.copy()
    frame.loc[EURO, "XGDDiffLast5"] = xgd[EURO.to_numpy()]
    frame.loc[EURO, "GoalsPerGameDiff"] = gpg[EURO.to_numpy()]
    features, train_rows = (C_FEATURES, EURO) if setup == "c" else (A_FEATURES, DOMESTIC)
    out = []
    for season in TEST:
        train = frame[train_rows & (frame["Season"] < season)]
        test = frame[EURO & (frame["Season"] == season)]
        out.append(predict(make_model().fit(train[features], train["FTR"]), test, features))
    return pd.concat(out).loc[euro_test.index]


cl_ref = european_proba(*SCOPES["S2"][:1], SCOPES["S1"][1])
euro_tune = euro_test[euro_test["Season"].isin(TUNE)]
c_tune_cl = {c: score(euro_tune, european_proba(S4[c], SCOPES["S1"][1]), cl_ref)["diff"] for c in C_GRID}
C_CL = min(c_tune_cl, key=c_tune_cl.get)
print("S4 c, European tune seasons (diff vs S2/S1):", {c: round(v, 5) for c, v in c_tune_cl.items()}, "-> c =", C_CL)

S4 c, European tune seasons (diff vs S2/S1): {100: -0.00132, 200: -0.00282, 400: -0.00198, 800: -0.00109} -> c = 200


In [11]:
cl_rows, cl_proba = [], {}
for xs in ["S1", "S2", "S3", "S4"]:
    for gs in ["S1", "S2", "S3"]:
        label = f"{xs}/{gs}"
        proba = cl_proba[label] = european_proba(xgd_scope(xs, C_CL), SCOPES[gs][1])
        on_cl = score(cl_test, proba, cl_ref)
        on_euro = score(euro_test, proba, cl_ref)
        cl_rows.append({
            "xGD/GPG": label, "CL log loss": on_cl["log loss"], "CL diff vs S2/S1": on_cl["diff"],
            "CL CI low": on_cl["CI low"], "CL CI high": on_cl["CI high"],
            "CL seasons better": seasons_better(cl_test["FTR"], proba.loc[cl_test.index][LABELS],
                                                cl_ref.loc[cl_test.index][LABELS], cl_test["Season"], LABELS),
            "Euro log loss": on_euro["log loss"], "Euro diff": on_euro["diff"],
            "Euro CI low": on_euro["CI low"], "Euro CI high": on_euro["CI high"],
        })
cl_grid = pd.DataFrame(cl_rows).set_index("xGD/GPG")
print(cl_grid.round(4).to_string())

         CL log loss  CL diff vs S2/S1  CL CI low  CL CI high  CL seasons better  Euro log loss  Euro diff  Euro CI low  Euro CI high
xGD/GPG                                                                                                                              
S1/S1         0.9256           -0.0019    -0.0085      0.0044                  3         0.9604     0.0009      -0.0036        0.0056
S1/S2         0.9251           -0.0024    -0.0109      0.0058                  2         0.9622     0.0027      -0.0031        0.0086
S1/S3         0.9258           -0.0017    -0.0097      0.0063                  2         0.9590    -0.0005      -0.0056        0.0046
S2/S1         0.9275            0.0000     0.0000      0.0000                  0         0.9595     0.0000       0.0000        0.0000
S2/S2         0.9235           -0.0040    -0.0091      0.0009                  3         0.9610     0.0015      -0.0023        0.0055
S2/S3         0.9265           -0.0010    -0.0045      0.0023 

Setup (a) (the domestic model applied as it is) with the same European scopes,
for reference - the S2/S1 and S3/S3 rows, against (a) at S2/S1:

In [12]:
a_ref = european_proba(SCOPES["S2"][0], SCOPES["S1"][1], setup="a")
a_rows = []
for label, (xs, gs) in {"S2/S1": ("S2", "S1"), "S3/S3": ("S3", "S3"), "S1/S1": ("S1", "S1"), "S2/S2": ("S2", "S2")}.items():
    proba = european_proba(xgd_scope(xs, C_CL), SCOPES[gs][1], setup="a")
    a_rows.append({"(a) at": label, **{f"CL {k}": v for k, v in score(cl_test, proba, a_ref).items()},
                   "Euro log loss": score(euro_test, proba, a_ref)["log loss"]})
print(pd.DataFrame(a_rows).set_index("(a) at").round(4).to_string())

        CL log loss  CL diff  CL CI low  CL CI high  CL better  Euro log loss
(a) at                                                                       
S2/S1        0.9388   0.0000     0.0000      0.0000      False         0.9685
S3/S3        0.9324  -0.0064    -0.0204      0.0072      False         0.9607
S1/S1        0.9330  -0.0058    -0.0124      0.0005      False         0.9670
S2/S2        0.9342  -0.0046    -0.0092     -0.0001       True         0.9654


### Honest selection for the CL

Twelve combinations were tried, so the winner on all four seasons is
flattered. Chosen on the tune seasons' European matches, reported on 24-25/25-26:

In [13]:
tune_loss = pd.Series({label: per_match_log_loss(euro_tune["FTR"], p.loc[euro_tune.index][LABELS], LABELS).mean()
                       for label, p in cl_proba.items()}).sort_values()
print("tune seasons, CL+EL+ECL log loss:"); print(tune_loss.round(4).to_string())

report_rows = []
for label in [tune_loss.index[0], "S3/S3", "S3/S2", "S4/S2"]:
    for rows, test_set in [(cl_test, "CL"), (euro_test, "CL+EL+ECL")]:
        r = rows[rows["Season"].isin(REPORT)]
        report_rows.append({"xGD/GPG": label, "test set": test_set, "n": len(r), **score(r, cl_proba[label], cl_ref)})
print(pd.DataFrame(report_rows).drop_duplicates(["xGD/GPG", "test set"]).set_index(["xGD/GPG", "test set"]).round(4).to_string())

tune seasons, CL+EL+ECL log loss:
S3/S3    0.9504
S4/S3    0.9519
S3/S1    0.9530
S2/S3    0.9542
S4/S1    0.9549
S2/S1    0.9578
S3/S2    0.9590
S4/S2    0.9593
S2/S2    0.9612
S1/S3    0.9615
S1/S1    0.9643
S1/S2    0.9677


                      n  log loss    diff  CI low  CI high  better
xGD/GPG test set                                                  
S3/S3   CL          378    0.9255 -0.0047 -0.0180   0.0081   False
        CL+EL+ECL  1062    0.9557 -0.0051 -0.0119   0.0015   False
S3/S2   CL          378    0.9271 -0.0031 -0.0158   0.0092   False
        CL+EL+ECL  1062    0.9559 -0.0049 -0.0116   0.0016   False
S4/S2   CL          378    0.9252 -0.0051 -0.0115   0.0013   False
        CL+EL+ECL  1062    0.9591 -0.0017 -0.0060   0.0027   False


## Conclusions (numbers as run on 2026-11-05)

**Domestic: keep S1/S1 (today's).** No combination is better: every one is
within ±0.0003 of S1/S1 on 9,530 domestic test matches, all CIs straddle 0
(S4 at its tuned c = 800: +0.00003 [-0.00071, +0.00078]). The European-club
subset (5,466 matches) shows nothing either - European games are too few of a
club's last five to move its form, as the plan expected.

**Rotation: neither ships.** `RestDaysDiff` +0.00014 [-0.00042, +0.00072],
`PostEuroMidweek` +0.00019 [-0.00022, +0.00062], both +0.00013 [-0.00050,
+0.00076]; no better on the European-club subset. (Domestic cups aren't in the
data, so rest days miss those games.)

**CL (setup (c)): no combination clears S2/S1 once selection is honest.**
- On all four test seasons the best CL log loss is S3/S2 (0.9201), -0.0074
  [-0.0183, +0.0037] against S2/S1: not clear of zero. S4/S2 (c = 200) is the
  one CI just clear on the CL (-0.0071 [-0.0139, -0.0001]), one of eleven
  comparisons, and only -0.0003 on CL+EL+ECL.
- On CL+EL+ECL the best is **S3/S3**, -0.0061 [-0.0119, -0.0002]: clear.
- Chosen on the tune seasons, S3/S3 is best (0.9504; S2/S1 0.9578); on the
  report seasons it is -0.0047 [-0.0180, +0.0081] on the CL and -0.0051
  [-0.0119, +0.0015] on CL+EL+ECL: favoured, not clear.
- So by the rule the fallback ships. The plan names two: §4B.5 says S2/S1, the
  Phase 5 gate says "the simplest". **Recommendation: S3/S3** - the simplest
  (each competition's own games, nothing joined across competitions), what the
  pipeline already builds, the scope notebook 22 chose setup (c) under, and the
  better one in every comparison above. Owner to confirm.
- Setup (a) shows the same ordering (S3/S3 0.9324 against S2/S1 0.9388 on the CL).